In [ ]:
# 訓練データとテストデータの画像を読み込む
# （サイズは縦横96pxにリサイズする）
import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(96, 96),
    label_mode="binary",
    batch_size=32,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(96, 96),
    label_mode="binary",
    batch_size=32,
    shuffle=False
)

Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.


In [6]:
# クラス名（['cat', 'dog'] ）
class_names = train_dataset.class_names
print("class_names:", class_names)

class_names: ['cat', 'dog']


In [ ]:
# 画像の水増しをする関数の定義
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label

In [4]:
# 画像の水増し処理の実行
train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)

In [8]:
# 水増ししたデータを訓練データに追加する
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

In [9]:
# データをシャッフルする
train_dataset = train_dataset.shuffle(32)

In [10]:
# MobileNetV2モデルを作成する
input_layer = tf.keras.Input(shape=(96, 96, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(96, 96, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [11]:
# Dense層を追加する
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')

In [12]:
# base_modelに先ほどのDense層を追加したモデルを作成する
model = tf.keras.Sequential([
    base_model,
    output_layer
])


In [13]:
# modelをcompileする
model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])

In [14]:
# modelに学習させる
model.fit(train_dataset, epochs=20)

Epoch 1/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 14s 106ms/step - accuracy: 0.8128 - loss: 0.3938
Epoch 2/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.9306 - loss: 0.1870
Epoch 3/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - accuracy: 0.9550 - loss: 0.1351
Epoch 4/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - accuracy: 0.9689 - loss: 0.1068
Epoch 5/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.9783 - loss: 0.0864
Epoch 6/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 94ms/step - accuracy: 0.9872 - loss: 0.0712
Epoch 7/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 95ms/step - accuracy: 0.9872 - loss: 0.0622
Epoch 8/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 96ms/step - accuracy: 0.9922 - loss: 0.0534
Epoch 9/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 101ms/step - accuracy: 0.9950 - loss: 0.0471
Epoch 10/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 6s 100ms/step - accuracy: 0.9950 - loss: 0.0419
Epoch 11/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 103ms/step - accuracy: 0.9967 - loss: 0.0371
Epoch 12/20
60/60 ━━━━━━━━━━━━━━━━━━━━ 7s 102ms/step -

In [15]:
# テストデータで分類を実行する
pred_data = model.predict(test_dataset)

4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 451ms/step


In [16]:
# 分類した結果を確認する
pred_data

array([[2.61863880e-03],
       [6.42103609e-04],
       [1.06591877e-04],
       [1.50199470e-04],
       [1.60500745e-03],
       [7.11203180e-03],
       [2.08079000e-04],
       [5.67649724e-04],
       [4.79796639e-04],
       [3.98929638e-04],
       [4.63359349e-04],
       [2.29232460e-02],
       [8.91525473e-04],
       [1.17670238e-01],
       [2.12592800e-04],
       [5.47274482e-04],
       [3.20723623e-01],
       [2.94837682e-03],
       [4.35624330e-04],
       [4.16346185e-04],
       [1.78055736e-04],
       [2.04315060e-03],
       [2.77299492e-04],
       [3.91048146e-04],
       [6.92054510e-01],
       [2.25533935e-04],
       [6.01710426e-03],
       [3.40733939e-04],
       [2.27741733e-01],
       [5.44761075e-03],
       [1.28537661e-03],
       [3.81057471e-01],
       [1.41890223e-06],
       [4.26519167e-04],
       [5.67538477e-03],
       [2.89014220e-01],
       [2.05566138e-02],
       [1.27093845e-05],
       [1.29273091e-03],
       [9.95445589e-05],


In [17]:
# evaluate()でモデルの性能を評価する
model.evaluate(test_dataset)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.9900 - loss: 0.0339


[0.03389878198504448, 0.9900000095367432]